In [3]:
import sys
import os
import joblib
import time
import pandas as pd
import numpy as np

from scipy.sparse import coo_matrix
import networkx as nx
from scipy.sparse.linalg import eigs

from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold, StratifiedKFold
from parsimony.algorithms.utils import Info

from matplotlib_venn import venn2, venn2_circles, venn2_unweighted
from matplotlib_venn import venn3, venn3_circles
from matplotlib import pyplot as plt
import seaborn as sns

import pickle

import NetSGCCA

In [6]:
def run(X, y, test_x, test_y, graphnet_L, l1, graphnet_lambda, g=None, groups = None, rna_=None, C = None):
    L_ = graphnet_L.copy()
    if (g is None):
        A_ = pd.DataFrame(np.diag(np.diag(graphnet_L.todense())) - graphnet_L, columns = rna.columns, index= rna.columns)
        g_  = nx.from_pandas_adjacency(A_)
    else : 
        g_ = g.copy()
    print("Staring pipe creation")
    # Create nodes
    scaler = NetSGCCA.PandasScaler()
    residualizer = NetSGCCA.CoxResiduals()
    feature_extractor = NetSGCCA.NetSGCCA(block_dict=groups, graphnet_L=L_,
                                                           graphnet_i=2,
                                                           max_inner_iter=10000, info=[Info.beta, Info.converged], n_comp = 1,
                                                           l1 = l1, 
                                                           graphnet_lambda = graphnet_lambda, C=C)
    predictor = NetSGCCA.SurvivalPrediction()
    
    # Create pipeline
    pipe = Pipeline(steps=[('scaler', scaler), ('residualizer', residualizer), ('feature_extractor', feature_extractor),
                           ('predictor', predictor)] , memory = None)
    
    # Run kfold
    print('Starting pipe fitting')
    kf = StratifiedKFold(n_splits=5)
    n_zeros = []
    weights = []
    full_weights = []
    scores= []
    for i, (train_index, test_index) in enumerate(kf.split(X, y['status'])):
        time0 = time.time()
        X_train, X_test = X.iloc[train_index], X.iloc[test_index]
        y_train, y_test = y.iloc[train_index], y.iloc[test_index]
        pipe.fit(X_train, y_train)
        selected = rna_.columns[(pipe['feature_extractor'].weights[2] != 0).reshape(1,  -1) [0]]
        weights.append(pipe['feature_extractor'].weights[2].reshape(1, -1))
        n_zeros.append(selected)
        scores.append(pipe.score(X_test, y_test))
        full_weights.append(pipe['feature_extractor'].weights)
        print('Fold {0} took {1} sec'.format(i,time.time()-time0))
    test_score = pipe.fit(X, y).score(test_x, test_y)
    del(pipe)
    # Extract sets
    print('Starting set extraction')
    all_intersected = set(n_zeros[0])
    for i in range(len(n_zeros)):
        all_intersected = all_intersected.intersection(set(n_zeros[i]))
    union = set(n_zeros[0])
    for i in range(len(n_zeros)):
        union = union.union(set(n_zeros[i]))
    sub_g_intersected = g_.subgraph(all_intersected)
    sub_g_union = g_.subgraph(union)
    sub_g_n_zeros = [g_.subgraph(d) for d in n_zeros]
    selected_genes_data  = {}
    for gene in union:
        data = {
            'number_of_folds' : len([d for d in n_zeros if gene in d]), 
            'degree_full_graph' : g_.degree(gene),
            'degree_intersection': sub_g_intersected.degree(gene) if gene in all_intersected else 0,
            'degree_union': sub_g_union.degree(gene), 
            'degree_in_folds': [sgu.degree(gene) for sgu in sub_g_n_zeros]
        }
        selected_genes_data[gene] = data
    return {'test_score':test_score, 'scores' : scores, 'full_weights': full_weights,
        'union': union, 'intersected': all_intersected, 'sub_graph_intersected':sub_g_intersected, "sub_g_union": sub_g_union,
        'non_zeros_sub_graphs' : sub_g_n_zeros, 'summary':  pd.DataFrame(selected_genes_data).T.copy(),  'weights': pd.DataFrame(np.concatenate(weights, axis = 0), columns = rna.columns)}

# Load data

In [5]:
source_dir = os.getenv('ROOTCGNET')

In [8]:
source_dir = '/volatile/Workspace/LocalCoxGNet/'

In [9]:
cnv = pd.read_csv(os.path.join(source_dir, "cnv.csv"), sep=";", index_col  =0)
mirna = pd.read_csv(os.path.join(source_dir, "mirna.csv"), sep=";", index_col  =0)
rna = pd.read_csv(os.path.join(source_dir, "PathwayCommons12.All.hgnc_subrna.csv"), sep=";", index_col  =0)
y = pd.read_csv(os.path.join(source_dir, "target.csv"), sep=";", index_col  =0)

In [10]:
X = [cnv, mirna, rna]
X_cat = pd.concat(X, axis = 1)
groups = {
    'group_{}'.format(i) :  list(X[i].columns) for i in range(len(X))
}

In [11]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X_cat, y,
                                                    stratify=y['status'], 
                                                    test_size=0.15)

# Load Graphs 

### Load pathway commons raw laplacian

In [12]:
L = pickle.load(open(os.path.join(source_dir, "PathwayCommons12.All.hgnc_laplacian.pkl"), "rb"))
L2 = coo_matrix(L)
A = pd.DataFrame(np.diag(np.diag(L2.todense())) - L2, columns = rna.columns, index= rna.columns)
g  = nx.from_pandas_adjacency(A)

In [13]:
l1 = [60.18928476066151,
   26,
   10.07491981459117,
   1]

### Load pathway commons normalized laplacian

In [14]:
L_normalied = pickle.load(open(os.path.join(source_dir, "PathwayCommons12.All.hgnc_normalized_laplacian.pkl"), "rb"))
L2_normalized = coo_matrix(L_normalied)

# 1. Comparing Raw and Normalized Laplacian (Pathway Commons)

We want to compare the raw pathway commons laplacian with its normalized version:
- We compare the number of selected variables for each laplacian
- We compare the stability of variable selection
- We compare degree distribution of selected gene in full graph the subgraph
- We compare weight distribution and grouping effects


### Compare number of selected variables

We vary $\lambda_g$ from $10^{-4}$ to $10^{3}$ and compare the number of selected genes

In [16]:
lambda_, _ = eigs(L2.astype('float'), k = 1)
lambda_ = 1/ lambda_[0].real
lambdas = [(10**k) * lambda_ for k in range(-4, 5)]
r = {}
for lam in lambdas:
    print(lam)
    data_pc = run(X_train, y_train,X_test, y_test, L2, l1, lam, g, groups, rna)
    r[lam] = data_pc.copy()

1.8238144368369302e-08
Staring pipe creation
Starting pipe fitting
Fold 0 took 6.298245429992676 sec
Fold 1 took 7.625305652618408 sec
Fold 2 took 7.226594686508179 sec
Fold 3 took 7.4005889892578125 sec
Fold 4 took 9.1876380443573 sec
Starting set extraction


In [ ]:
lambda_, _ = eigs(L2_normalized.astype('float'), k = 1)
lambda_ = 1/ lambda_[0].real
lambdas = [(10**k) * lambda_ for k in range(-4, 5)]
r_normalized = {}
for lam in lambdas :
    print(lam)
    data_pc = run(X_train, y_train,X_test, y_test, L2_normalized, l1, lam, g, groups, rna)
    r_normalized[lam] = data_pc.copy()

In [ ]:
distribution = []

In [ ]:
lambda_, _ = eigs(L2_normalized.astype('float'), k = 1)
lambda_ = 1/ lambda_[0].real
for k, v in r_normalized.items():
    graphss =  v['non_zeros_sub_graphs']
    for ggs in graphss:
        distribution.append({
            'lambda': round(np.log10(k/lambda_)),
            'number_of_selected_genes': len(ggs.nodes),
            'type' : 'normalized'
        })

In [ ]:
lambda_, _ = eigs(L2.astype('float'), k = 1)
lambda_ = 1/ lambda_[0].real
for k, v in r.items():
    graphss =  v['non_zeros_sub_graphs']
    for ggs in graphss:
        distribution.append({
            'lambda': round(np.log10(k/lambda_)),
            'number_of_selected_genes': len(ggs.nodes),
            'type' : 'raw'
        })

In [ ]:
distribution = pd.DataFrame(distribution)
distribution = distribution[distribution['lambda'] < 5]

In [ ]:
_, ax = plt.subplots(figsize=(4,4))
ax = sns.pointplot(ax=ax, x="lambda", y="number_of_selected_genes", hue="type",
                   data=distribution,  palette=['blue','red'])
ax.set_xlabel(r'log10($\gamma_\mathcal{G}$)')
plt.savefig('images/selected_number.png')

Figure shows the the number of selected genes by the raw and normalized laplacian is similar. This confirms literary results: with a fixed L1 penalty, graphnet selects more variables as its coefficient increases

### Compare the stability of variable selection

In [ ]:
selected_lambda_raw = list(r.keys())[6]
selected_lambda_normalized = list(r_normalized.keys())[6]

In [ ]:
summary_complete = r[selected_lambda_raw]['summary']
summary_complete['type'] = 'raw'
summary_normalized = r_normalized[selected_lambda_normalized]['summary']
summary_normalized['type'] = 'normalized'

results3 = pd.concat([summary_complete, summary_normalized])
results3 = results3.reset_index()
_, ax = plt.subplots(figsize=(4,4))
ax = sns.countplot(ax=ax, data=results3, x="number_of_folds", hue="type", palette=['red','blue'])
ax.set_xticks([0,1,2,3,4])
ax.set_xticklabels(['1_fold','2_folds','3_folds','4_folds','5_folds'])
ax.set_xlabel('')
plt.savefig('images/reocurrance.png')

The graph shows, with a fixed $\lambda_g$, the histogram of the number of times (folds) each gene has selected. It shows that, for the raw graph, many genes were selected only once. On the other hand, with the normaized graph, many genes were selected all 5 times.

In [ ]:
p1 = len(r[selected_lambda_raw]['intersected']) / len(r[selected_lambda_raw]['union'])
p2 = len(r_normalized[selected_lambda_normalized]['intersected']) / len(r_normalized[selected_lambda_normalized]['union'])
p1 = round(p1, 2)
p2 = round(p2, 2)
print('The ratio of genes selected 5 times among all selected genes was %s for the raw graph, and %s for the normalized graph' % (p1,p2))

# Stability

In [ ]:
stability = []

In [ ]:
lambda_, _ = eigs(L2_normalized.astype('float'), k = 1)
lambda_ = 1/ lambda_[0].real
for k, v in r_normalized.items():
    stability.append({
            'lambda': round(np.log10(k/lambda_)),
            'stability': len(v['intersected']) / len(v['union']),
            'type' : 'normalized'
        })

In [ ]:
lambda_, _ = eigs(L2.astype('float'), k = 1)
lambda_ = 1/ lambda_[0].real
for k, v in r.items():
    stability.append({
            'lambda': round(np.log10(k/lambda_)),
            'stability': len(v['intersected']) / len(v['union']),
            'type' : 'raw'
        })

In [ ]:
stability = pd.DataFrame(stability)
stability = stability[stability['lambda'] < 5]

In [ ]:
_, ax = plt.subplots(figsize=(4,4))
ax = sns.pointplot(ax=ax, x="lambda", y="stability", hue="type",
                   data=stability, palette=['blue','red'])
ax.set_xlabel(r'log10($\gamma_\mathcal{G}$)')
plt.savefig("images/stability.png")

### Compare degree distribution of selected gene in full graph the subgraph

In [ ]:
graphs = r[selected_lambda_raw]['non_zeros_sub_graphs'][0]
degree_sequence = sorted((g.degree(n) for n, d in graphs.degree()), reverse=True)
dff = [{'degree': np.floor(np.log(x+1)/np.log(2)), 'fold': 0} for x in degree_sequence]
for i in range(1, 5):
    graphs = r[selected_lambda_raw]['non_zeros_sub_graphs'][i]
    degree_sequence = sorted((g.degree(n) for n, d in graphs.degree()), reverse=True)
    dff.extend([{'degree': np.floor(np.log(x+1)/np.log(2)) if x > 0 else 0, 'fold': i} for x in degree_sequence])
dff = pd.DataFrame(dff)

In [ ]:
graphs = r_normalized[selected_lambda_normalized]['non_zeros_sub_graphs'][0]
degree_sequence = sorted((g.degree(n) for n, d in graphs.degree()), reverse=True)
dff2 = [{'degree': np.floor(np.log(x+1)/np.log(2)), 'fold': 0} for x in degree_sequence]
for i in range(1, 5):
    graphs = r_normalized[selected_lambda_normalized]['non_zeros_sub_graphs'][i]
    degree_sequence = sorted((g.degree(n) for n, d in graphs.degree()), reverse=True)
    dff2.extend([{'degree': np.floor(np.log(x+1)/np.log(2)), 'fold': i} for x in degree_sequence])
dff2 = pd.DataFrame(dff2)

In [ ]:
degrees_test = pd.DataFrame(g.degree, columns = ['gene', 'degree'])
degrees_test['degree2'] = degrees_test['degree'].apply(lambda x: np.floor(np.log(x+1)/np.log(2)))

In [ ]:
dff['degree2'] = dff['degree'].apply(lambda x: str(int(x)))
dff2['degree2'] = dff2['degree'].apply(lambda x: str(int(x)))
degrees_test['degree3'] = degrees_test['degree2'].apply(lambda x: str(int(x)))


In [ ]:
dff['fold2'] = dff['fold']
dff2['fold2'] = dff2['fold']
dff['fold'] = dff['fold2'].apply(lambda x: 'fold_' + str(x))
dff2['fold'] = dff2['fold2'].apply(lambda x: 'fold_' + str(x))

In [ ]:
palette = sns.color_palette('flare', as_cmap=True)
fig, axs = plt.subplots(1,2, figsize=(18,4))
sns.histplot(data=dff.sort_values(by='degree'), x="degree2", hue="fold", legend = True, ax=axs[0], common_norm=False,
    stat='density', multiple='dodge', palette='flare', hue_order=['fold_0','fold_1','fold_2','fold_3','fold_4'])
axs[0].set_title('Genes selected with raw')
sns.histplot(data=dff2.sort_values(by='degree'), x="degree2", hue="fold", legend = True, ax=axs[1],common_norm=False, stat='density'
             , multiple='dodge', palette='flare', hue_order=['fold_0','fold_1','fold_2','fold_3','fold_4'])
axs[1].set_title('Genes selected with normalized')
#
nax = fig.add_axes(axs[0].get_position(), frameon=False)
sns.kdeplot( ax=nax, data=degrees_test.sort_values(by='degree2'), x="degree3", legend=False, bw=0.2, linewidth=2, color='black')
nax.set_xlim(axs[0].get_xlim())
nax.set_ylim((0,0.25))
axs[0].set_ylim((0,0.25))
nax.tick_params(left=False, bottom=False)
nax.set_ylabel('')
nax.set_xlabel('')

nax = fig.add_axes(axs[1].get_position(), frameon=False)
sns.kdeplot( ax=nax, data=degrees_test.sort_values(by='degree2'), x="degree3", legend=False, bw=0.2, linewidth=2, color='black')
nax.set_xlim(axs[1].get_xlim())
nax.set_ylim((0,0.25))
axs[1].set_ylim((0,0.25))
nax.tick_params(left=False, bottom=False)
nax.set_ylabel('')
nax.set_xlabel('')


#axs[0].plot(axs[2].get_lines())
#axs[2].set_title('All genes')
axs[0].set_xlabel('log2(degree)')
axs[1].set_xlabel('log2(degree)')
#axs[2].set_xlabel('log2(degree)')
plt.savefig('images/degree_dist.png')

The graphs show the degree distribution of selected genes in the pathway commons graph. We can see a similatiry between the selected genes with the raw and normalized laplacian. These two distributions follow a similar pattern as the distribution of all the genes. This suggests that the graph doesnt favor high or low density nodes, except for the isolated nodes, which are underrepresented.

In [ ]:
graphs = r[selected_lambda_raw]['non_zeros_sub_graphs'][0]
degree_sequence = sorted((d for n, d in graphs.degree()), reverse=True)
dff = [{'degree': np.floor(np.log(x+1)/np.log(2)), 'fold': 0} for x in degree_sequence]
for i in range(1, 5):
    graphs = r[selected_lambda_raw]['non_zeros_sub_graphs'][i]
    degree_sequence = sorted((d for n, d in graphs.degree()), reverse=True)
    dff.extend([{'degree': np.floor(np.log(x+1)/np.log(2)), 'fold': i} for x in degree_sequence])
dff = pd.DataFrame(dff)

In [ ]:
graphs = r_normalized[selected_lambda_normalized]['non_zeros_sub_graphs'][0]
degree_sequence = sorted((d for n, d in graphs.degree()), reverse=True)
dff2 = [{'degree': np.floor(np.log(x+1)/np.log(2)), 'fold': 0} for x in degree_sequence]
for i in range(1, 5):
    graphs = r_normalized[selected_lambda_normalized]['non_zeros_sub_graphs'][i]
    degree_sequence = sorted((d for n, d in graphs.degree()), reverse=True)
    dff2.extend([{'degree': np.floor(np.log(x+1)/np.log(2)), 'fold': i} for x in degree_sequence])
dff2 = pd.DataFrame(dff2)

In [ ]:
dff['degree2'] = dff['degree'].apply(lambda x: str(int(x)))
dff2['degree2'] = dff2['degree'].apply(lambda x: str(int(x)))

In [ ]:
dff['fold2'] = dff['fold']
dff2['fold2'] = dff2['fold']
dff['fold'] = dff['fold2'].apply(lambda x: 'fold_' + str(x))
dff2['fold'] = dff2['fold2'].apply(lambda x: 'fold_' + str(x))

In [ ]:
_, axs = plt.subplots(1,2, figsize=(18,4))
sns.histplot(data=dff.sort_values(by='degree'), x="degree2", hue="fold", legend = True, ax=axs[0], stat='density'
             , multiple='dodge', palette='flare', hue_order=['fold_0','fold_1','fold_2','fold_3','fold_4'], common_norm=False)
axs[0].set_title('Raw')
sns.histplot(data=dff2.sort_values(by='degree'), x="degree2", hue="fold", legend = True, ax=axs[1], stat='density'
             , multiple='dodge', palette='flare', hue_order=['fold_0','fold_1','fold_2','fold_3','fold_4'], common_norm=False)
axs[1].set_title('Normalized')
axs[0].set_xlabel('log2(degree)')
axs[1].set_xlabel('log2(degree)')
axs[0].set_ylim((0,0.7))
axs[1].set_ylim((0,0.7))
plt.savefig('images/degree_dist_sub.png')

The graphs show the degree distribution of selected genes in the subgraph of selected genes. There is a shift towards the lower values. This suggest that the model doesnt select communities

### Compare weight distribution and grouping effects

In [ ]:
def extract_weights(r_, l):
    W =r_[l]['weights']
    R = {}
    ll = []
    for i in range(5):
        gg= r_[l]['non_zeros_sub_graphs'][i]
        length = dict(nx.all_pairs_dijkstra_path_length(gg))
        length = pd.DataFrame(length)
        ww = W.loc[i, list(gg.nodes)]
        for n1 in range(len(length.columns)-1):
            for n2 in range(n1+1, len(length.columns)):
                gene1, gene2 =length.columns[n1], length.columns[n2]
                d= pd.DataFrame(length).loc[gene1, gene2]
                if(pd.isna(d) or d > 5): continue
                ll.append({'length': d, 'dist': np.abs(ww[gene1] - ww[gene2]), 'fold': i, 'contrib': ww[gene1] *  ww[gene2] > 0, 'sign' : ww[gene1] > 0})
    return pd.DataFrame(ll)

In [ ]:
def extract_weights_abs(r_, l):
    W =r_[l]['weights']
    R = {}
    ll = []
    for i in range(5):
        gg= r_[l]['non_zeros_sub_graphs'][i]
        length = dict(nx.all_pairs_dijkstra_path_length(gg))
        length = pd.DataFrame(length)
        ww = W.loc[i, list(gg.nodes)]
        for n1 in range(len(length.columns)-1):
            for n2 in range(n1+1, len(length.columns)):
                gene1, gene2 =length.columns[n1], length.columns[n2]
                d= pd.DataFrame(length).loc[gene1, gene2]
                if(pd.isna(d) or d > 5): continue
                ll.append({'length': d, 'dist': np.abs(np.abs(ww[gene1]) - np.abs(ww[gene2])), 'fold': i})
    return pd.DataFrame(ll)

In [ ]:
def extract_weights2(r_, l):
    W =r_[l]['weights']
    R = {}
    ll = []
    for i in range(5):
        gg= r_[l]['non_zeros_sub_graphs'][i]
        #length = dict(nx.all_pairs_dijkstra_path_length(gg))
        #length = pd.DataFrame(length)
        ww = W.loc[i, list(gg.nodes)]
        for n1 in gg.nodes():
            d= g.degree(n1)
            w = ww[n1]
            ll.append({'degree': d, 'weight': w, 'fold': i})
    return pd.DataFrame(ll)

In [ ]:
D_weights_diff_normalized = extract_weights(r_normalized, selected_lambda_normalized)
D_weights_diff_raw = extract_weights(r, selected_lambda_raw)

In [ ]:
D_weights_diff_normalized_abs = extract_weights_abs(r_normalized, selected_lambda_normalized)
D_weights_diff_raw_abs = extract_weights_abs(r, selected_lambda_raw)

In [ ]:
D_weights_normalized = extract_weights2(r_normalized, selected_lambda_normalized)
D_weights_normalized['type'] = 'normalized'
D_weights_raw = extract_weights2(r, selected_lambda_raw)
D_weights_raw['type'] = 'raw'
D_weights = pd.concat([D_weights_raw, D_weights_normalized])
D_weights['log2degree'] = D_weights['degree'].apply(lambda x: np.floor(np.log(x+1)/ np.log(2)))
D_weights = D_weights[D_weights['log2degree']>0]
D_weights['sign'] = D_weights['weight'].apply(lambda x: x>0)

In [ ]:
_, axs = plt.subplots(1,2, figsize=(12,4))
sns.boxplot(x="log2degree", y="weight", data=D_weights[D_weights['type'] == 'raw'],  showfliers = False, ax= axs[0])
sns.boxplot(x="log2degree", y="weight", data=D_weights[D_weights['type'] == 'normalized'],  showfliers = False, ax= axs[1])


In [ ]:
_, axs = plt.subplots(1,2, figsize=(15,3))
D_weights2= D_weights[D_weights['log2degree']< 11]
sns.boxplot(x="log2degree", y="weight", data=D_weights2[D_weights2['type'] == 'raw'],  showfliers = False, ax= axs[0], whis=0.1)
sns.boxplot(x="log2degree", y="weight", data=D_weights2[D_weights2['type'] == 'normalized'],  showfliers = False, ax= axs[1], whis=0.1)
axs[0].set_title('Raw')
axs[1].set_title('Normalized')
plt.savefig('images/weight_dist.png')

In [ ]:
D_weights

The box-plot shows the weight distribution according to the degree. We can see that all selected genes have a similar weights. The same pattern can be exhibited for the normalized and the raw laplacian, except that the normalized laplanian has lower weights. 

We can also see that all selected genes have positive weights. This can be explain by the fact that very few genes are selected without a graph, thus the graph spreads the weights on similar genes in terms of correlation.

In [ ]:
grouped_raw = D_weights_diff_raw.groupby(['length', 'fold'])
grouped_raw = grouped_raw.apply(lambda x: x.sample(frac=0.2))
grouped_raw['type'] = 'raw'
grouped_normalized = D_weights_diff_normalized.groupby(['length', 'fold'])
grouped_normalized = grouped_normalized.apply(lambda x: x.sample(frac=0.2))
grouped_normalized['type'] = 'normalized'

grouped_mean = pd.concat([grouped_raw, grouped_normalized])

In [ ]:
grouped_normalized['dist3'].max()

In [ ]:
grouped_mean = pd.concat([grouped_raw, grouped_normalized])

In [ ]:
fig, axs = plt.subplots(1,1, figsize=(4,4))
sns.pointplot(x="length", y="dist", data = grouped_mean, ax= axs, palette=['blue', 'red'], hue="type")

axs.set_xlabel('Djikstra distance in the graph')
axs.set_ylabel('Absolute value of weight difference')
fig.tight_layout()
plt.savefig('images/grouping.png')

In [ ]:
_, axs = plt.subplots(1,2, figsize=(12,4))
sns.pointplot(x="length", y="dist", data = grouped_raw[grouped_raw['contrib'] * grouped_raw['sign']], ax= axs[0])
sns.pointplot(x="length", y="dist", data = grouped_normalized[grouped_normalized['contrib'] * grouped_normalized['sign']], ax= axs[0])

In [ ]:
_, axs = plt.subplots(1,2, figsize=(12,4))
sns.pointplot(x="length", y="dist", data = grouped_raw[~grouped_raw['contrib']], ax= axs[0])
sns.pointplot(x="length", y="dist", data = grouped_normalized[~grouped_normalized['contrib']], ax= axs[1])

In [ ]:
grouped_raw = D_weights_diff_raw_abs.groupby(['length', 'fold'])
grouped_raw = grouped_raw.apply(lambda x: x.sample(frac=0.9))
grouped_raw['type'] = 'raw'
grouped_normalized = D_weights_diff_normalized_abs.groupby(['length', 'fold'])
grouped_normalized = grouped_normalized.apply(lambda x: x.sample(frac=0.9))
grouped_normalized['type'] = 'normalized'

grouped_mean = pd.concat([grouped_raw, grouped_normalized])

In [ ]:
_, axs = plt.subplots(1,2, figsize=(12,4))
sns.pointplot(x="length", y="dist", data = grouped_raw, ax= axs[0])
sns.pointplot(x="length", y="dist", data = grouped_normalized, ax= axs[1])

For each pair of selected genes, we calculated the length of the shortest path between them, and the absolute difference between their weights. The plot shows that, the closer the genes (in the graph), the closer their weights. The same pattern can be observed using the raw or normalized graph.

# 2. Comparison between different graphs

### Pathway commons vs KEGG vs MSIGDB

In [ ]:
L_kegg = pickle.load(open(os.path.join(source_dir, "PathwayCommons12.kegg.hgnc_normalized_laplacian.pkl"), "rb"))
L2_kegg = coo_matrix(L_kegg)
lambda_, _ = eigs(L2_kegg.astype('float'), k = 1)
lambda_ = 100/ lambda_[0].real
r_kegg =  run(X_train, y_train,X_test, y_test, L2_kegg, l1, lambda_, g, groups, rna)

In [ ]:
L_kegg = pickle.load(open(os.path.join(source_dir, "PathwayCommons12.kegg.hgnc_normalized_laplacian.pkl"), "rb"))
L_msigdb = pickle.load(open(os.path.join(source_dir, "PathwayCommons10.msigdb.hgnc_normalized_laplacian.pkl"), "rb"))

In [ ]:
L_msigdb = pickle.load(open(os.path.join(source_dir, "PathwayCommons10.msigdb.hgnc_normalized_laplacian.pkl"), "rb"))
L2_msigdb = coo_matrix(L_msigdb)
lambda_, _ = eigs(L2_msigdb.astype('float'), k = 1)
lambda_ = 100/ lambda_[0].real
r_msigdb =  run(X_train, y_train,X_test, y_test, L2_msigdb, l1, lambda_, g, groups, rna)

In [ ]:
_, axs = plt.subplots(1,2, figsize=(12,4))
out0 = venn3([r_normalized[selected_lambda_normalized]['union'], r_msigdb['union'], r_kegg['union']], ax=axs[0],
      set_labels = ('Normalized PC', 'Normalized Msigdb', 'Normalized KEGG'), set_colors=['blue','red','green'])

out0.subset_labels[6].set_visible(False)
out0.subset_labels[2].set_visible(False)

axs[0].text(0.1,-0.1, '241')
axs[0].text(0,0.3, '306')
out = venn3([r_normalized[selected_lambda_normalized]['intersected'], r_msigdb['intersected'], r_kegg['intersected']], ax=axs[1],
      set_labels = ('Normalized PC', 'Normalized Msigdb', 'Normalized KEGG'), set_colors=['blue','red','green'])
out.subset_labels[6].set_visible(False)
out.subset_labels[2].set_visible(False)

axs[1].text(0.1,-0.1, '122')
axs[1].text(0,0.3, '122')
plt.savefig('images/venn_msigdb.png')

In [ ]:
out.subset_labels

Using the KEGG graph with its normalized laplacian, all genes selected in the 5 folds and at least once are all selected by the PC normalized graph.

This is the same result found using the MSIGDB graph. However here fewer genes are selected. This is explained by the fact that KEGG graph is smaller and less dense.

### Pathway commons vs Permutated PC

We permuted the graph 10 times and compared the intersection of genes selected at least once, twice ...

In [ ]:
lambda_, _ = eigs(L2_kegg.astype('float'), k = 1)
lambda_ = 100/ lambda_[0].real
r_permutations_kegg = {}
for x in range(10) :
    rna_permuted = rna[np.random.permutation(rna.columns)]
    X_permutation = [cnv, mirna, rna_permuted]
    X_cat_permutation = pd.concat(X_permutation, axis = 1)
    groups_permuted = {
        'group_{}'.format(i) :  list(X_permutation[i].columns) for i in range(len(X_permutation))
    }
    data_pc = run(X_cat_permutation, y, L2_kegg, l1, lambda_, g, groups_permuted, rna_ = rna_permuted)
    r_permutations_kegg[x] = data_pc.copy()
    i1 = data_pc['intersected']

In [ ]:
dfs = []
truth_summary = r_kegg['summary']
for pp in range(10):
    for fd in range(1,6):
        idx = r_permutations_kegg[pp]['summary'].index[(r_permutations_kegg[pp]['summary'].loc[:,'number_of_folds']>=fd)]
        idx_truth = truth_summary.index[(truth_summary.loc[:,'number_of_folds']>=fd)]
        dfs.append({
            "permutation": pp, 
            "folds": fd, 
            "len_perm" : len(idx),
            "len_truth": len(idx_truth), 
            "len_inter": round(len(set(idx).intersection(set(idx_truth)))/len(idx), 2)
        })
dfs = pd.DataFrame(dfs)

In [ ]:
ax= sns.pointplot(x="folds", y="len_inter", data=dfs)

In [ ]:
X_cat_permutation.loc[X_train.index]

In [ ]:
lambda_, _ = eigs(L2_normalized.astype('float'), k = 1)
lambda_ = 100/ lambda_[0].real
r_permutations = {}
for x in range(10) :
    rna_permuted = rna[np.random.permutation(rna.columns)]
    X_permutation = [cnv, mirna, rna_permuted]
    X_cat_permutation = pd.concat(X_permutation, axis = 1)
    groups_permuted = {
        'group_{}'.format(i) :  list(X_permutation[i].columns) for i in range(len(X_permutation))
    }
    X_cat_permutation_train = X_cat_permutation.loc[X_train.index]
    X_cat_permutation_test = X_cat_permutation.loc[X_test.index]
    data_pc = run(X_cat_permutation_train, y_train,X_cat_permutation_test, y_test, L2_normalized, l1, lambda_, g, groups_permuted, rna_ = rna_permuted)
    r_permutations[x] = data_pc.copy()

In [ ]:
dfs = []
r_normalizedx = r_normalized[selected_lambda_normalized]
truth_summary = r_normalizedx['summary']
for pp in range(10):
    for fd in range(1,6):
        idx = r_permutations[pp]['summary'].index[(r_permutations[pp]['summary'].loc[:,'number_of_folds']>=fd)]
        idx_truth = truth_summary.index[(truth_summary.loc[:,'number_of_folds']>=fd)]
        dfs.append({
            "permutation": pp, 
            "folds": fd, 
            "len_perm" : len(idx),
            "len_truth": len(idx_truth), 
            "len_inter": round(len(set(idx).intersection(set(idx_truth)))/len(set(idx).union(set(idx_truth))), 2)
        })
dfs = pd.DataFrame(dfs)

In [ ]:
ax = sns.boxplot(x="folds", y="len_inter", data=dfs[dfs['folds'].isin([1,5])])
ax.set_xticks([0,1],['Candidate selected genes','Stable selected genes'])
ax.set_xlabel('')
ax.set_ylabel('Intersection over union')
plt.savefig('images/permutations.png')

In [ ]:
ax= sns.pointplot(x="folds", y="len_inter", data=dfs)
ax.tick_params(axis="x",direction="in", pad=-15)
ax.set_ylim((0.69,0.78))
ax.set_ylabel('Intersection over union')

fig = plt.gcf()
nax = fig.add_axes(ax.get_position(), frameon=False)
nax.set_xticks([1,5],['Candidate \nselected genes','Stable\nselected genes'],rotation=0)
nax.set_yticks([])
nax.set_xlim((0.5,5.5))

The figure shows that percentage of genes selected by the permuted graph and the original graph are highly overlapped. This shows that primary attribute that determines gene selection is not information embedded in the graph but its structure and topology for the number of genes selected and the RGCCA is responsible of which genes are selected. 

### Pathway commins vs PC - MCN

We removed, from the pathway commons, direct, undirected and all links between selected nodes

In [ ]:
removed2 = {}
edges_in = set(g.edges(r_normalizedx['union']))
edges_out = set(g.edges()) 

for n in ['union','intersected']:
    edges_all = g.edges(r_normalizedx[n])
    edges_sub = g.subgraph(r_normalizedx[n]).edges
    edges_diff = set(edges_all) - set(edges_sub)
    for ename, elist in zip(['all','sub','diff'], [edges_all,edges_sub,edges_diff]):
        print(n, ename)
        sub_g  = g.copy()
        sub_g.remove_edges_from(elist)
        L2_sub_g = coo_matrix(nx.normalized_laplacian_matrix(sub_g))
        lambda_, _ = eigs(L2_sub_g.astype('float'), k = 1)
        lambda_ = (10**2)/ lambda_[0].real
        r_sub_g = run(X_train, y_train, X_test, y_test, L2_sub_g, l1, lambda_, g, groups, rna)
        removed2[n+"_"+ename] = r_sub_g

In [ ]:
removed2 = {}
edges_in = set(g.edges(r_normalizedx['union']))
edges_out = set(g.edges)  - edges_in
edges_in = list(edges_in)
edges_out = list(edges_out)
NN = 193720086.0
for p in np.linspace(0, 1, 10):
    g_copy = g.copy()
    to_del_in = np.random.choice(len(edges_in), size = int(len(edges_in) *p), replace=False)
    to_del_out = np.random.choice(len(np.array(edges_out)), int(len(edges_out) *p), replace=False)
    print(to_del_in)
    print(to_del_out)
    if(len(to_del_in) > 0):
        g_copy.remove_edges_from([edges_in[i] for i in to_del_in])
    if(len(to_del_out) > 0):
        g_copy.remove_edges_from([edges_out[i] for i in to_del_out])
    L2_sub_g = coo_matrix(nx.normalized_laplacian_matrix(g_copy))
    lambda_, _ = eigs(L2_sub_g.astype('float'), k = 1)
    lambda_ = (10**2)/ lambda_[0].real
    r_sub_g = run(X_train, y_train, X_test, y_test, L2_sub_g, l1, lambda_, g_copy, groups, rna)
    removed2[p] = {'res' : r_sub_g, "p": p, "density": len(g_copy.edges) / NN}

In [ ]:
remove2_data = []
ref = {'union' : removed2[list(removed2.keys())[0]]['res']['union'],
  'intersected' : removed2[list(removed2.keys())[0]]['res']['intersected']}
for d in removed2.keys():
    for n in ['union', 'intersected']:
        b = 822 if n == 'union' else  423
        a = 1022 if n == 'union' else 408
        remove2_data.append({'d':round(d, 2), 
            'Number_of_selected_genes' : len(removed2[d]['res'][n]),
            'Density': removed2[d]['density'],
            'd2': round(1-d, 2), 
             'type' : 'Candidate' if n == 'union' else 'Stable',
            'IOU' : len(ref[n].intersection(removed2[d]['res'][n])) / len(ref[n].union(removed2[d]['res'][n])),
             'IN':       len(ref[n].intersection(removed2[d]['res'][n])) / len((removed2[d]['res'][n]))})

In [ ]:
remove2_data = pd.DataFrame(remove2_data)

In [ ]:
_, ax = plt.subplots(figsize=(4,4))
ax = sns.pointplot(ax=ax, x="d", y="Number_of_selected_genes", hue="type",
                   data=remove2_data,  palette=['orange','blue'], scilimits=(4, 4))
ax.set_xlabel('% of removed edges')

In [ ]:
d = []
for x, v in removed2.items():
    rr = {'removed': x}
    for t in ['union', 'intersected']:
        idx = v[t]
        idx_truth = r_normalizedx[t]
        rr.update({
                "len_perm_"+t : len(idx),
                "len_truth_"+t: len(idx_truth), 
                "len_inter_"+t: round(len(set(idx).intersection(set(idx_truth)))/len(idx) if len(idx) > 0 else 0, 2)
            })
    d.append(rr)
removed_df = pd.DataFrame(d)

# Correlation analysis

In [ ]:
corr = rna.corr()

In [ ]:
corr_intersected = corr.loc[r_normalizedx['intersected'], r_normalizedx['intersected']].values
corr_intersected[np.eye(len(corr_intersected)) == 1] =  np.nan
idx= np.triu_indices(len(corr_intersected))
corr_intersected = corr_intersected[idx[0], idx[1]]
corr_intersected = corr_intersected[~np.isnan(corr_intersected)]
corr_intersected = pd.DataFrame(corr_intersected, columns=["values"])

corr_union = corr.loc[r_normalizedx['union'], r_normalizedx['union']].values
corr_union[np.eye(len(corr_union)) == 1] =  np.nan
idx= np.triu_indices(len(corr_union))
corr_union = corr_union[idx[0], idx[1]]
corr_union = corr_union[~np.isnan(corr_union)]
corr_union = pd.DataFrame(corr_union, columns=["values"])

corr_full = corr.copy().values
corr_full[np.eye(len(corr_full)) == 1] =  np.nan
idx= np.triu_indices(len(corr_full))
corr_full = corr_full[idx[0], idx[1]]
corr_full = corr_full[~np.isnan(corr_full)]
corr_full = pd.DataFrame(corr_full, columns=["values"])

In [ ]:
fig , axs = plt.subplots(1, 3, figsize=(10,3))
sns.histplot(data = corr_intersected, x="values", ax = axs[0])
axs[0].set_title("Stable selected genes")
axs[0].set_yticklabels(['']*len(axs[0].get_yticklabels()))
axs[0].set_ylabel('')
sns.histplot(data = corr_union, x="values", ax = axs[1])
axs[1].set_title("Candidate selected genes")
axs[1].set_yticklabels(['']*len(axs[1].get_yticklabels()))
axs[1].set_ylabel('')
sns.histplot(data = corr_full, x="values", ax = axs[2])
axs[2].set_title("All genes")
axs[2].set_yticklabels(['']*len(axs[2].get_yticklabels()))
axs[2].set_ylabel('')
fig.tight_layout()
fig.savefig('images/correlations.png')

# Enrichement Analysis

In [ ]:
import gseapy as gs

In [ ]:
enrichr = gs.enrichr(list(r_normalized[selected_lambda_normalized]['union']), 'MSigDB_Oncogenic_Signatures')
enrichr.results[enrichr.results['P-value']  < 0.05].sort_values('Adjusted P-value')

In [ ]:
enrichr = gs.enrichr(list(r_kegg['intersected']), 'MSigDB_Oncogenic_Signatures')
enrichr.results[enrichr.results['P-value']  < 0.05].sort_values('Adjusted P-value')

In [ ]:
enrichr = gs.enrichr(list(r_msigdb['intersected']), 'MSigDB_Oncogenic_Signatures')
enrichr.results[enrichr.results['P-value']  < 0.05].sort_values('Adjusted P-value')

In [ ]:
enrichr = gs.enrichr(list(r[selected_lambda_raw]), 'MSigDB_Oncogenic_Signatures')
enrichr.results[enrichr.results['P-value']  < 0.05].sort_values('Adjusted P-value')